# RAG Chatbot — Attention Is All You Need

A minimal Retrieval-Augmented Generation pipeline over the "Attention Is All You Need" paper (Vaswani et al., 2017).

**Pipeline:** PDF download → text extraction → chunking → embeddings (sentence-transformers) → FAISS index → semantic retrieval → Claude-generated answer grounded in retrieved context.


## Setup

In [1]:
import os, io, textwrap, urllib.request
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

## Task 1: Document Loading and Chunking

In [2]:
PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = "attention.pdf"

MIRRORS = [PDF_URL, "https://arxiv.org/pdf/1706.03762v7", "https://arxiv.org/pdf/1706.03762.pdf"]

def download_pdf():
    last_err = None
    for url in MIRRORS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read()
            with open(PDF_PATH, "wb") as f:
                f.write(data)
            print(f"Downloaded {len(data):,} bytes from {url}")
            return
        except Exception as e:
            last_err = e
            print(f"Failed {url}: {e}")
    raise RuntimeError(f"All mirrors failed: {last_err}")

download_pdf()

Downloaded 2,215,244 bytes from https://arxiv.org/pdf/1706.03762


In [3]:
reader = PdfReader(PDF_PATH)
pages = [p.extract_text() or "" for p in reader.pages]
raw_text = "\n".join(pages)
print(f"Pages: {len(pages)}")
print(f"Total characters: {len(raw_text):,}")
print("Preview:")
print(raw_text[:400])

Pages: 15
Total characters: 39,611
Preview:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G


In [4]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        window = words[i:i + chunk_size]
        if not window:
            break
        chunks.append(" ".join(window))
        if i + chunk_size >= len(words):
            break
    return chunks

chunks = chunk_text(raw_text, chunk_size=180, overlap=30)
print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk [12]:\n{chunks[12][:600]}")

Total chunks: 41

Sample chunk [12]:
this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in 
